## 環境設置

In [1]:
try:
    import openseespy.opensees as ops  # noqa: F401
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops  # noqa: F401

# Case-07:反應譜分析 + ATC-40 容量譜法(CSM)

依 [ROADMAP.md](../ROADMAP.md) 規劃,把 Case-06.5 已經驗證過的
V-Δ 側推曲線,轉換成反應譜可以查表的語言,找出真正的地震需求
(性能點)。

**先講清楚整體邏輯,不是只端公式**:我們的建築是好幾層樓的多自由度
(MDOF)系統,但反應譜是拿單一自由度(SDOF)系統算出來的——這裡
要做的事,是把 MDOF 結構「濃縮」成一個等效的 SDOF 系統,才能拿
反應譜查地震需求,再把結果換算回真實的屋頂位移。這一步是連接
「已經做完的 MDOF 側推」跟「地震需求」中間唯一缺的橋樑。

**沿用哪些已驗證的東西**:
- V-Δ 側推曲線、$K_e$、$V_y$、$D_y$(0.6$V_y$ 割線法):Case-06.5
- 構架幾何、質量概估:Case-04.5/06.5
- 設計反應譜:最初的法規計算 notebook(`seismic_design_2story_8col.ipynb`)

## 第 1 課:振態分析——把 MDOF 濃縮成等效 SDOF

**這裡不是跳過就能做的東西**——特徵值分析要用結構**未開裂、未
降伏的初始彈性狀態**,不能直接沿用 Case-06.5 那個已經進入非線性
的纖維斷面模型,兩者代表的物理狀態不同,混用會有問題。這裡改用
均質彈性斷面重新建模,專門用來抓自然週期跟振型。

In [2]:
L_bay = 6.0
h1 = h2 = 3.5
b = h = 0.40
b_beam, h_beam = 0.3, 0.5
Ib = b_beam*h_beam**3/12
A_beam = b_beam*h_beam
E_rc = 2.463e7

# 質量: 沿用一路使用的重力載重概估(每根柱147.6kN軸力來源的載重), 換算成質量
W_per_col = 147.60   # kN
W_floor = 2*W_per_col   # 這個構架2根柱共同扛的樓層重量
m_floor = W_floor/9.81   # 質量(tonnes)


def build_frame_elastic_for_modal():
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    Ic = h**4/12
    A_col = h**2
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)

    ops.mass(3, m_floor/2, 1e-9, 1e-9)
    ops.mass(4, m_floor/2, 1e-9, 1e-9)
    ops.mass(5, m_floor/2, 1e-9, 1e-9)
    ops.mass(6, m_floor/2, 1e-9, 1e-9)

    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn', 1, 1, 3, A_col, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, A_col, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, A_col, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, A_col, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 5, 3, 4, A_beam, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 6, 5, 6, A_beam, E_rc, Ib, 1)


build_frame_elastic_for_modal()
n_modes = 2
eigenvalues = ops.eigen(n_modes)

import numpy as np
periods = [2*np.pi/ev**0.5 for ev in eigenvalues]
print(f"樓層質量估計: m_floor = {m_floor:.3f} tonnes(每層)")
print(f"自然週期: T1={periods[0]:.4f}s, T2={periods[1]:.4f}s")

phi_1F = ops.nodeEigenvector(3, 1, 1)
phi_roof = ops.nodeEigenvector(5, 1, 1)
print(f"\n第一振型(原始特徵向量): 1F={phi_1F:.5f}, roof={phi_roof:.5f}")

assert periods[0] > periods[1], "基本振態(T1)應該是最長的週期"
print("\n[PASS] 特徵值分析完成, T1為最長週期(基本振態)")

樓層質量估計: m_floor = 30.092 tonnes(每層)
自然週期: T1=0.4703s, T2=0.1422s

第一振型(原始特徵向量): 1F=0.07896, roof=0.16431

[PASS] 特徵值分析完成, T1為最長週期(基本振態)


## 第 2 課:振態參與係數 $\Gamma$ 與等效振態質量 $M^*$

$\Gamma$、$M^*$ 是把 MDOF 濃縮成 SDOF 的關鍵係數。**這裡有個容易
搞混的地方要先講清楚**:$\Gamma$ 這個數字本身會隨振型正規化方式
改變(不是一個絕對值),但 $M^*=\Gamma \times \Sigma(m_i\phi_i)$
這個組合是正規化無關的(可以證明數學上等於 $[\Sigma(m_i\phi_i)]^2
/\Sigma(m_i\phi_i^2)$,不管振型怎麼縮放都一樣)——這裡直接用
OpenSeesPy 算出來的原始特徵向量,不特別正規化,重點看 $M^*$ 這個
最終有意義的量。

In [3]:
masses = np.array([m_floor, m_floor])
phis = np.array([phi_1F, phi_roof])

L_modal = np.sum(masses*phis)
M_modal = np.sum(masses*phis**2)
Gamma = L_modal/M_modal
M_star = Gamma*L_modal

M_total = np.sum(masses)
alpha1 = M_star/M_total

phi_1F_norm = phi_1F/phi_roof   # 正規化(屋頂=1), 方便報告呈現與後續pushover位移對應

print(f"第一振型(正規化, 屋頂=1.0): 1F={phi_1F_norm:.4f}, roof=1.0000")
print(f"振態參與係數 Gamma = {Gamma:.4f}")
print(f"等效振態質量 M* = {M_star:.3f} tonnes")
print(f"總質量 = {M_total:.3f} tonnes")
print(f"有效質量比 alpha1 = M*/M_total = {alpha1:.4f} ({alpha1*100:.1f}%)")

assert 0.7 < alpha1 < 1.0, "規則的2層樓構架, 第一振態有效質量比通常落在70~100%之間"
print("\n[PASS] 有效質量比落在合理範圍, 確認第一振態確實主導這個結構的反應")
print("(alpha1接近90%代表用第一振態近似整個結構的反應是合理的簡化)")

第一振型(正規化, 屋頂=1.0): 1F=0.4806, roof=1.0000
振態參與係數 Gamma = 7.3204
等效振態質量 M* = 53.588 tonnes
總質量 = 60.183 tonnes
有效質量比 alpha1 = M*/M_total = 0.8904 (89.0%)

[PASS] 有效質量比落在合理範圍, 確認第一振態確實主導這個結構的反應
(alpha1接近90%代表用第一振態近似整個結構的反應是合理的簡化)


## 第 3 課:有效週期 $T_e$——為什麼比彈性週期 $T_i$ 長

用 Case-06.5 驗證過的 $K_e$(0.6$V_y$ 割線勁度,已經反映結構軟化後
的狀態),算出來的 $T_e$ 應該比彈性特徵值分析的 $T_1$ 長——**這是
地震工程裡熟悉的物理現象(結構軟化、週期拉長),不是算錯**。

In [4]:
Ke_case065 = 5576.59   # Case-06.5驗證過的0.6Vy割線勁度(kN/m)
Ti = periods[0]

Te_from_Mstar = 2*np.pi*(M_star/Ke_case065)**0.5

print(f"Ti(彈性特徵值分析) = {Ti:.4f}s")
print(f"Te(用M*和Ke算, Te=2π√(M*/Ke)) = {Te_from_Mstar:.4f}s")
print(f"Te/Ti = {Te_from_Mstar/Ti:.2f}")

assert Te_from_Mstar > Ti, "有效週期應該比彈性週期長(結構軟化後週期拉長)"
print("\n[PASS] Te > Ti, 符合結構軟化後週期拉長的物理預期")
print("這個比值本身反映了結構從彈性到降伏這段期間, 整體勁度衰減了多少")

Ti(彈性特徵值分析) = 0.4703s
Te(用M*和Ke算, Te=2π√(M*/Ke)) = 0.6159s
Te/Ti = 1.31

[PASS] Te > Ti, 符合結構軟化後週期拉長的物理預期
這個比值本身反映了結構從彈性到降伏這段期間, 整體勁度衰減了多少


## 總結表

In [5]:
print("="*55)
print("Case-07 第1階段: 振態分析總結")
print("="*55)
print(f"{'T1(彈性)':<20}{Ti:.4f} s")
print(f"{'T2':<20}{periods[1]:.4f} s")
print(f"{'Gamma':<20}{Gamma:.4f}")
print(f"{'M*(等效振態質量)':<20}{M_star:.3f} tonnes")
print(f"{'alpha1(有效質量比)':<20}{alpha1:.1%}")
print(f"{'Te(有效週期)':<20}{Te_from_Mstar:.4f} s")
print()
print("Case-07第1階段 [PASS] -- 振態分析完成, 可進入DCM/CSM需求解法")

Case-07 第1階段: 振態分析總結
T1(彈性)              0.4703 s
T2                  0.1422 s
Gamma               7.3204
M*(等效振態質量)          53.588 tonnes
alpha1(有效質量比)       89.0%
Te(有效週期)            0.6159 s

Case-07第1階段 [PASS] -- 振態分析完成, 可進入DCM/CSM需求解法
